# Aula 19 — Correlação, covariância e causalidade

Laboratório reproduzível da disciplina **02-statistics**.

Vamos comparar Pearson e Spearman, visualizar o quarteto de Anscombe e produzir associações espúrias por agregação, seleção e busca múltipla.

**Ambiente:** Python 3.10+; NumPy ≥ 1.24; pandas ≥ 2.0; SciPy ≥ 1.11; Matplotlib ≥ 3.7.  
**Seed global:** `20260907`.


## Protocolo metodológico

- Todos os dados, exceto o quarteto clássico incorporado no código, são sintéticos.
- Correlações descrevem as amostras; não serão chamadas de efeitos causais.
- O bootstrap preserva os pares `(x, y)`.
- Grupos e mecanismos de seleção permanecem visíveis.
- As asserções verificam o padrão que cada simulação foi criada para demonstrar.


In [ ]:
# Dependências: numpy>=1.24, pandas>=2.0, scipy>=1.11, matplotlib>=3.7
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats

SEED = 20260907
rng = np.random.default_rng(SEED)

print(f"Seed: {SEED}")
print(f"NumPy {np.__version__} | pandas {pd.__version__} | SciPy {scipy.__version__} | Matplotlib {matplotlib.__version__}")


## 1. Linear, monotônica e em U

Pearson resume linearidade; Spearman resume monotonicidade por postos. Nenhum dos dois detecta necessariamente uma relação em U.


In [ ]:
rng_formas = np.random.default_rng(SEED + 1)
n = 800

x_linear = rng_formas.uniform(-3, 3, n)
y_linear = 0.9 * x_linear + rng_formas.normal(0, 0.8, n)

x_monotona = rng_formas.uniform(0, 4, n)
y_monotona = np.exp(x_monotona) + rng_formas.normal(0, 1.5, n)

x_u = rng_formas.uniform(-3, 3, n)
y_u = x_u**2 + rng_formas.normal(0, 0.6, n)

cenarios = {
    "linear": (x_linear, y_linear),
    "monotônica curva": (x_monotona, y_monotona),
    "forma em U": (x_u, y_u),
}

linhas = []
for nome, (x, y) in cenarios.items():
    pearson = stats.pearsonr(x, y)
    spearman = stats.spearmanr(x, y)
    linhas.append({
        "relação": nome,
        "Pearson_r": pearson.statistic,
        "Pearson_p": pearson.pvalue,
        "Spearman_rho": spearman.statistic,
        "Spearman_p": spearman.pvalue,
    })

correlacoes_formas = pd.DataFrame(linhas)
exibicao = correlacoes_formas.copy()
for coluna in ["Pearson_r", "Spearman_rho"]:
    exibicao[coluna] = exibicao[coluna].map(lambda v: f"{v:.6f}")
for coluna in ["Pearson_p", "Spearman_p"]:
    exibicao[coluna] = exibicao[coluna].map(lambda v: f"{v:.3e}")
print(exibicao.to_string(index=False))

assert correlacoes_formas.loc[0, "Pearson_r"] > 0.85
assert correlacoes_formas.loc[1, "Spearman_rho"] > correlacoes_formas.loc[1, "Pearson_r"]
assert abs(correlacoes_formas.loc[2, "Pearson_r"]) < 0.10
assert abs(correlacoes_formas.loc[2, "Spearman_rho"]) < 0.10


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, (nome, (x, y)), cor in zip(axes, cenarios.items(), ["#2563eb", "#0f766e", "#d97706"]):
    ax.scatter(x, y, s=12, alpha=0.45, color=cor)
    linha = correlacoes_formas.loc[correlacoes_formas["relação"] == nome].iloc[0]
    ax.set(title=f"{nome}\nr={linha.Pearson_r:.2f}; ρs={linha.Spearman_rho:.2f}", xlabel="X", ylabel="Y")
fig.suptitle("O mesmo coeficiente não serve para toda forma", y=1.03)
plt.tight_layout()
plt.show()


## 2. Intervalo de confiança de Pearson

Comparamos o intervalo pela transformação de Fisher, fornecido pelo SciPy, e o bootstrap percentil. No bootstrap, os índices são compartilhados entre `x` e `y`; sortear colunas separadamente destruiria a associação.


In [ ]:
resultado_pearson = stats.pearsonr(x_linear, y_linear)
ic_fisher_obj = resultado_pearson.confidence_interval(confidence_level=0.95)
ic_fisher = np.array([ic_fisher_obj.low, ic_fisher_obj.high])

B_BOOT = 5_000
rng_boot = np.random.default_rng(SEED + 2)
indices = rng_boot.integers(0, n, size=(B_BOOT, n))
r_boot = np.empty(B_BOOT)
for b, idx in enumerate(indices):
    r_boot[b] = np.corrcoef(x_linear[idx], y_linear[idx])[0, 1]
ic_boot = np.quantile(r_boot, [0.025, 0.975])

print(f"r observado = {resultado_pearson.statistic:.6f}")
print(f"IC 95% Fisher = [{ic_fisher[0]:.6f}; {ic_fisher[1]:.6f}]")
print(f"IC 95% bootstrap = [{ic_boot[0]:.6f}; {ic_boot[1]:.6f}]")

assert ic_fisher[0] < resultado_pearson.statistic < ic_fisher[1]
assert ic_boot[0] < resultado_pearson.statistic < ic_boot[1]
assert np.max(np.abs(ic_fisher - ic_boot)) < 0.02


## 3. Quarteto de Anscombe

Os quatro conjuntos têm onze pares e resumos praticamente iguais. Os dados estão incorporados para eliminar dependência de rede; a fonte é Anscombe (1973).


In [ ]:
x_comum = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], dtype=float)
quarteto = {
    "I": (x_comum, np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68])),
    "II": (x_comum, np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])),
    "III": (x_comum, np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73])),
    "IV": (np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], dtype=float), np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])),
}

resumos = []
for nome, (x, y) in quarteto.items():
    inclinacao, intercepto = np.polyfit(x, y, 1)
    resumos.append({
        "conjunto": nome,
        "media_x": x.mean(), "var_x": x.var(ddof=1),
        "media_y": y.mean(), "var_y": y.var(ddof=1),
        "r": stats.pearsonr(x, y).statistic,
        "intercepto": intercepto, "inclinacao": inclinacao,
    })

anscombe_resumo = pd.DataFrame(resumos)
print(anscombe_resumo.round(4).to_string(index=False))

assert anscombe_resumo["media_x"].max() - anscombe_resumo["media_x"].min() < 1e-12
assert anscombe_resumo["r"].max() - anscombe_resumo["r"].min() < 0.002
assert anscombe_resumo["inclinacao"].max() - anscombe_resumo["inclinacao"].min() < 0.002


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)
x_reta = np.linspace(3, 20, 100)
for ax, (nome, (x, y)) in zip(axes.ravel(), quarteto.items()):
    inclinacao, intercepto = np.polyfit(x, y, 1)
    ax.scatter(x, y, s=45, color="#2563eb")
    ax.plot(x_reta, intercepto + inclinacao * x_reta, color="#dc2626", linewidth=1.8)
    ax.set(title=f"Conjunto {nome}", xlim=(3, 20), ylim=(3, 14), xlabel="X", ylabel="Y")
fig.suptitle("Quarteto de Anscombe: resumos semelhantes, estruturas diferentes", y=1.01)
plt.tight_layout()
plt.show()


## 4. Paradoxo de Simpson

Em cada nível de dificuldade, `X` e `Y` têm correlação positiva. Tarefas mais difíceis, porém, recebem valores maiores de `X` e têm interceptos menores de `Y`. A mistura inverte o sinal.


In [ ]:
rng_simpson = np.random.default_rng(SEED + 4)
nomes_grupo = ["fácil", "média", "difícil"]
medias_x = [1.0, 3.0, 5.0]
interceptos = [8.0, 5.0, 2.0]
n_grupo = 120

partes = []
for nome, media_x, intercepto in zip(nomes_grupo, medias_x, interceptos):
    x = rng_simpson.normal(media_x, 0.45, n_grupo)
    y = intercepto + 0.7 * x + rng_simpson.normal(0, 0.45, n_grupo)
    partes.append(pd.DataFrame({"dificuldade": nome, "X": x, "Y": y}))
dados_simpson = pd.concat(partes, ignore_index=True)

corr_por_grupo = dados_simpson.groupby("dificuldade", sort=False).apply(
    lambda d: stats.pearsonr(d["X"], d["Y"]).statistic,
    include_groups=False,
)
corr_agregada = stats.pearsonr(dados_simpson["X"], dados_simpson["Y"]).statistic

print("Correlação por dificuldade:")
print(corr_por_grupo.round(6).to_string())
print(f"Correlação agregada = {corr_agregada:.6f}")

assert (corr_por_grupo > 0.45).all()
assert corr_agregada < -0.70


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
cores = {"fácil": "#2563eb", "média": "#0f766e", "difícil": "#d97706"}
for nome, d in dados_simpson.groupby("dificuldade", sort=False):
    ax.scatter(d["X"], d["Y"], s=18, alpha=0.55, color=cores[nome], label=f"{nome}: r={corr_por_grupo[nome]:.2f}")
    b1, b0 = np.polyfit(d["X"], d["Y"], 1)
    faixa = np.linspace(d["X"].min(), d["X"].max(), 30)
    ax.plot(faixa, b0 + b1 * faixa, color=cores[nome])
b1_total, b0_total = np.polyfit(dados_simpson["X"], dados_simpson["Y"], 1)
faixa_total = np.linspace(dados_simpson["X"].min(), dados_simpson["X"].max(), 50)
ax.plot(faixa_total, b0_total + b1_total * faixa_total, color="#111827", linewidth=2.5, linestyle="--", label=f"agregado: r={corr_agregada:.2f}")
ax.set(title="Paradoxo de Simpson", xlabel="X", ylabel="Y")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Viés de colisor por seleção

`habilidade` e `indicação` são geradas independentemente. Ambas aumentam a chance de seleção. Ao analisar apenas selecionados, condicionamos no colisor e fabricamos uma correlação negativa.


In [ ]:
rng_colisor = np.random.default_rng(SEED + 5)
n_pop = 100_000
habilidade = rng_colisor.normal(size=n_pop)
indicacao = rng_colisor.normal(size=n_pop)
ruido_selecao = rng_colisor.normal(0, 0.35, n_pop)
selecionado = habilidade + indicacao + ruido_selecao > 1.5

r_pop = stats.pearsonr(habilidade, indicacao).statistic
r_sel = stats.pearsonr(habilidade[selecionado], indicacao[selecionado]).statistic

print(f"Proporção selecionada = {selecionado.mean():.6f}")
print(f"Correlação na população = {r_pop:.6f}")
print(f"Correlação entre selecionados = {r_sel:.6f}")

assert abs(r_pop) < 0.02
assert r_sel < -0.55


In [ ]:
rng_plot = np.random.default_rng(SEED + 50)
idx_pop = rng_plot.choice(n_pop, 3_000, replace=False)
idx_sel = rng_plot.choice(np.flatnonzero(selecionado), 3_000, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
axes[0].scatter(habilidade[idx_pop], indicacao[idx_pop], s=8, alpha=0.3, color="#64748b")
axes[0].set(title=f"População: r={r_pop:.2f}", xlabel="Habilidade", ylabel="Indicação")
axes[1].scatter(habilidade[idx_sel], indicacao[idx_sel], s=8, alpha=0.3, color="#dc2626")
axes[1].set(title=f"Somente selecionados: r={r_sel:.2f}", xlabel="Habilidade")
plt.tight_layout()
plt.show()


## 6. Correlação espúria por busca múltipla

Criamos 500 *features* independentes de um alvo. Ainda assim, dezenas podem ter p-value bruto menor que 0,05 e a maior correlação absoluta parecer interessante.


In [ ]:
rng_mult = np.random.default_rng(SEED + 6)
n_amostra = 120
m_features = 500
X = rng_mult.normal(size=(n_amostra, m_features))
alvo = rng_mult.normal(size=n_amostra)

resultados = [stats.pearsonr(X[:, j], alvo) for j in range(m_features)]
r_features = np.array([res.statistic for res in resultados])
p_features = np.array([res.pvalue for res in resultados])
indice_max = int(np.argmax(np.abs(r_features)))
n_brutos = int(np.sum(p_features < 0.05))
limiar_bonf = 0.05 / m_features
n_bonf = int(np.sum(p_features < limiar_bonf))

print(f"Features com p bruto < 0,05: {n_brutos} de {m_features}")
print(f"Maior |r| = {abs(r_features[indice_max]):.6f} (feature {indice_max})")
print(f"Menor p bruto = {p_features.min():.9f}")
print(f"Limiar Bonferroni = {limiar_bonf:.7f}; rejeições = {n_bonf}")

assert 10 <= n_brutos <= 45
assert abs(r_features[indice_max]) > 0.25
assert n_bonf == 0


## 7. Checagens finais

As verificações abaixo resumem as mensagens principais. Elas não convertem os exemplos em evidência causal; apenas garantem reprodutibilidade dos mecanismos simulados.


In [ ]:
assert abs(correlacoes_formas.loc[2, "Pearson_r"]) < 0.10
assert anscombe_resumo["r"].max() - anscombe_resumo["r"].min() < 0.002
assert (corr_por_grupo > 0).all() and corr_agregada < 0
assert abs(r_pop) < 0.02 and r_sel < -0.55
assert n_brutos > 0 and n_bonf == 0

print("Todas as verificações foram aprovadas.")
print(f"Relação em U: Pearson={correlacoes_formas.loc[2, 'Pearson_r']:.6f}; Spearman={correlacoes_formas.loc[2, 'Spearman_rho']:.6f}")
print(f"Simpson: correlações internas mínimas={corr_por_grupo.min():.6f}; agregada={corr_agregada:.6f}")
print(f"Colisor: população={r_pop:.6f}; selecionados={r_sel:.6f}")
print(f"Busca múltipla: {n_brutos} resultados brutos; {n_bonf} após Bonferroni")


## Desafios

1. Aumente o ruído da relação monotônica e acompanhe Pearson e Spearman.
2. Remova o ponto de alta alavancagem do conjunto IV de Anscombe e recalcule `r`.
3. Altere os interceptos do exemplo de Simpson até a correlação agregada trocar de sinal.
4. Mude o limiar de seleção do colisor e observe a correlação induzida.
5. Repita a busca múltipla com 5.000 *features* e aplique Holm ou BH da Aula 18.

Antes de usar dados reais, escreva a pergunta associativa ou causal e identifique a unidade experimental.
